# Other measures: GenPop wave 2 vs Enriched wave 1 (PHQ / GAD / BAARS)

Full stepwise pipeline (baseline → metric search → scalar continuation with metric fallback → strict report) for the six non-HiTOP scales, comparing **genpop recontact (wave 2) vs enriched wave 1**. Both waves come from the gridall files -- the recontact cohort, in which every subject has both waves (GP n=398, EN n=255; note these are smaller samples than the grid1st data used by the main pipelines, so tests are less powered). The two groups are independent subjects (different samples), so the standard multigroup assumptions hold. Outputs are isolated in `data/cfa_other_gp2_en1/`. On re-run, the notebook reuses its own completed baseline rows (seed-identical). Run headless with `./notebooks/run_overnight.sh NB_2_cfa_as_reg_other_gp2_en1.ipynb`.

# Prep

## Import stuff

In [1]:
from pathlib import Path
import pandas as pd
pd.set_option('max_colwidth', 100)
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix
import itertools
import scipy.stats as st
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
#import seaborn as sns
#from matplotlib import pyplot as plt
#%matplotlib inline
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 500)
import rpy2
#import pingouin as pg
from itertools import combinations
import openpyxl
from contextlib import redirect_stdout
import random
import math
import platform

## Some magical magic to make the R stuff work

In [2]:
import os
os.environ["OMP_NUM_THREADS"], os.environ["OPENBLAS_NUM_THREADS"], os.environ["MKL_NUM_THREADS"] 

('1', '1', '1')

In [3]:
# The rpy2/R session and CFA machinery now live in the hitop_cfa package.
# Importing it starts the embedded R session: it sets the BLAS threading
# env vars (if unset) and loads base, utils, lavaan, and the patched semTools.
import hitop_cfa
from hitop_cfa.r_env import (ro, rbase, utils, lavaan, semtools,
                             RRuntimeError, pandas2ri, localconverter)

import rpy2.ipython.html
rpy2.ipython.html.init_printing()

# Paths

In [4]:
# paths where to save preprocessed data files
log_dir = Path('./log')
log_dir.mkdir(exist_ok=True)
dat_dir = Path('../data/')
val_dir = dat_dir / 'ValSample'
fin_dir = dat_dir / 'finaldata'
cfa_dir = dat_dir / 'cfa_other_gp2_en1'
cfa_dir.mkdir(exist_ok=True)
path_save_val = fin_dir / 'dat_val.csv'
path_save_dat_gp_grid1st_norecontact = fin_dir / 'dat_gp_grid1st_norecontact.csv'
path_save_dat_en_grid1st_norecontact = fin_dir / 'dat_en_grid1st_norecontact.csv'
path_save_dat_gp_grid1st_full = fin_dir / 'dat_gp_grid1st_full.csv'
path_save_dat_en_grid1st_full = fin_dir / 'dat_en_grid1st_full.csv'
path_save_dat_gp_gridall_full = fin_dir / 'dat_gp_gridall_full.csv'
path_save_dat_en_gridall_full = fin_dir / 'dat_en_gridall_full.csv'
# path_save_dat_gp_gridall_recontact = '../../data/finaldata/dat_gp_gridall_recontact.csv'
# path_save_dat_en_gridall_recontact = '../../data/finaldata/dat_en_gridall_recontact.csv'
# helped file for cfa
helpfile_dir = cfa_dir / 'temp'
helpfile_dir.mkdir(exist_ok=True, parents=True)
path_to_helpfile = helpfile_dir / 'cfa_temp.csv'
path_to_cogmood_questions = dat_dir / 'cogmood_questions.csv'
path_to_item_lookup = val_dir / 'Internalizing-Somatoform Items_DW.xlsx'

In [5]:
import datetime
import traceback

# Overnight robustness: every long loop below wraps its per-scale work in
# try/except and calls this on failure, so ONE bad scale cannot kill the
# whole run. Errors are printed AND appended (with tracebacks) to
# cfa_dir/run_errors.log; run_error_records collects them for the summary
# cell at the end of the notebook.
run_error_records = []


def record_run_error(stage, scale, exc):
    stamp = datetime.datetime.now().isoformat(timespec='seconds')
    msg = f"[{stamp}] ERROR in {stage} for scale {scale!r}: {exc!r}"
    print(msg)
    run_error_records.append(dict(stage=stage, scale=scale, error=repr(exc)))
    with open(cfa_dir / 'run_errors.log', 'a') as ef:
        ef.write(msg + '\n')
        ef.write(traceback.format_exc() + '\n')

## Count how many cpus I have, then decide how many I want to use; define how many iterations for CFA (decrease for debugging)

In [6]:
def get_architecture():
    # Get the raw machine architecture string
    arch = platform.machine().lower()
    
    if "arm" in arch or "aarch" in arch:
        return "ARM"
    elif "x86" in arch or "amd" in arch or "i386" in arch or "i686" in arch:
        return "x86"
    else:
        return f"Unknown ({arch})"

In [7]:
total_cpus = os.cpu_count()
# account for hyperthreading
arch = get_architecture()
if arch == 'ARM':
    cpus_to_use = total_cpus - 2
else:
    cpus_to_use = total_cpus // 2 -1
global cpus_to_use
print(f"\nGoing to use {cpus_to_use} CPUs for CFA heavy-lifting\n")

num_iter = 1000
global num_iter


Going to use 14 CPUs for CFA heavy-lifting



## SET SEEDS !!!!!!!!!!

In [8]:
#rngkind = "L'Ecuyer-CMRG"
random.seed(12345)

In [9]:
ro.r('RNGkind(kind = "L\'Ecuyer-CMRG")')
ro.r('set.seed(12345)')

<rpy2.rinterface_lib.sexp.NULLType object at 0x13194c9d0> [0]

### TEST THE SEEDS!!!!!!!!

In [10]:
for i in range(5):
    print(random.random())
# after kernel restart, this should be 
# 0.41661987254534116
# 0.010169169457068361
# 0.8252065092537432
# 0.2986398551995928
# 0.3684116894884757

0.41661987254534116
0.010169169457068361
0.8252065092537432
0.2986398551995928
0.3684116894884757


In [11]:
ro.r('rnorm(5)')
# after kernel restart, this should be 
# -1.457850350316457	-0.45246126454182867	0.3650586371545244	-1.57091128601566	1.1419085835874878

-1.457850350316457,-0.45246126454182867,0.3650586371545244,-1.57091128601566,1.1419085835874878


### I'M ALSO SETTING THE SAME SEEDS EVERY TIME I RUN THE HELPED CFA FUNCTION, JUST IN CASE!!!!!

# Functions

## CFA helper functions

In [12]:
from hitop_cfa import (
    build_luts,
    check_hitop_ids,
    cfa_helper_func,
    run_specific_cfa,
    do_three_way_cfa_stepwise_mi,
    do_three_way_cfa_stepwise_scalar,
    do_stepwise_scalar_from_metric_run,
    load_metric_run,
    exhaustive_cfa_ablations,
    set_seeds,
    silence_r,
)

# item-text lookups (was load_item_lookup + inline lut construction)
_luts = build_luts(path_to_item_lookup, path_to_cogmood_questions)
item_lookup = _luts['item_lookup']
item_lut = _luts['item_lut']
phq_lut = _luts['phq_lut']
gad_lut = _luts['gad_lut']
baars_lut = _luts['baars_lut']

# these scales' item texts live in the instrument-specific luts;
# item ids (phq_N, gad_N, inattention_N, ...) are unique across them
item_lut = {**phq_lut, **gad_lut, **baars_lut['inattention'],
            **baars_lut['hyperactivity'], **baars_lut['impulsivity'],
            **baars_lut['sct']}

/Users/nielsond/code/hitop_val/scalar/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/scalar/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/scalar/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/scalar/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


## CFA wrapper functions

# Run Main Code

## Load preprocessed data and concatinate

In [13]:
# Other measures, genpop recontact (wave 2) vs enriched wave 1.
# The gridall files hold the recontact cohort (every row has both waves).
# Wave-2 frames take the *_recontact item columns renamed to the plain
# item ids; the group column ('whichdata') is set explicitly per wave.
from hitop_cfa.scales import OTHER_SCALES as _os
_other_items = [i for f in _os.values()
                for i in f.split('=~', 1)[1].strip().split(' + ')]
data_gp_all = pd.read_csv(path_save_dat_gp_gridall_full)
data_en_all = pd.read_csv(path_save_dat_en_gridall_full)

def _wave_frame(df, wave, label):
    if wave == 1:
        out = df[_other_items].copy()
    else:
        out = df[[f'{i}_recontact' for i in _other_items]].copy()
        out.columns = _other_items
    out['whichdata'] = label
    return out

group1 = _wave_frame(data_gp_all, 2, 'gp_t2')
group2 = _wave_frame(data_en_all, 1, 'en_t1')
data_pair = pd.concat([group1, group2])
print(data_pair.whichdata.value_counts())

datasets_runspecific = {'GP2_EN1': data_pair}
datasets_stepwise = {'gp2_en1': data_pair}


# Invariance analyses (measEq / Wu & Estabrook 2016 ladder)

All invariance models are generated by `semTools::measEq.syntax` with `ID.cat = "Wu.Estabrook.2016"` and `ID.fac = "std.lv"` — plain `group.equal` shortcuts are vacuous at scalar/strict for ordinal indicators, and marker identification is underidentified under Wu–Estabrook (see `HANDOFF_measeq_fix.md`). The level ladder is:

**configural → thresholds → metric → scalar → strict**

where metric = thresholds + loadings, scalar = + intercepts, strict = + residuals. `assert_level_adds_df` runs before every permutation delta test, so a vacuous comparison raises instead of silently passing. There is no marker item under `std.lv`.

## Original scale formulas

In [14]:
# the six non-HiTOP scales; single source of truth in hitop_cfa.scales
from hitop_cfa.scales import OTHER_SCALES
orig_items = dict(OTHER_SCALES)
orig_items

{'phq_sum': 'phq_sum=~phq_1 + phq_2 + phq_3 + phq_4 + phq_5 + phq_6 + phq_7 + phq_8',
 'gad_sum': 'gad_sum =~gad_1 + gad_2 + gad_3 + gad_4 + gad_5 + gad_6 + gad_7',
 'baars_inattention_sum': 'baars_inattention_sum =~inattention_1 + inattention_2 + inattention_3 + inattention_4 + inattention_5 + inattention_6 + inattention_7 + inattention_8 + inattention_9',
 'baars_hyperactivity_sum': 'baars_hyperactivity_sum =~hyperactivity_1 + hyperactivity_2 + hyperactivity_3 + hyperactivity_4 + hyperactivity_5',
 'baars_impulsivity_sum': 'baars_impulsivity_sum =~impulsivity_1 + impulsivity_2 + impulsivity_3 + impulsivity_4',
 'baars_sct_sum': 'baars_sct_sum =~sct_1 + sct_2 + sct_3 + sct_4 + sct_5 + sct_6 + sct_7 + sct_8 + sct_9'}

## Baseline: 5-level ladder over the six scales (self-resuming)

No other pipeline tests this wave comparison, so there is nothing to reuse on a first run; on a RE-run the notebook reuses its own completed `orig_cfa_res.csv` rows (seed-identical: `cfa_helper_func` reseeds at every scale × pair call). The stepwise stages always run fresh.

In [15]:
# Self-resume: reuse this notebook's OWN completed baseline rows from a
# previous run (no other pipeline tests this wave comparison). Scales
# missing from the record get a fresh ladder.
threeway_csv = cfa_dir / 'orig_cfa_res.csv'
reused = pd.DataFrame()
if threeway_csv.exists():
    reused = pd.read_csv(threeway_csv)
    reused = reused[(reused['pair'] == 'GP2_EN1')
                    & reused['scale'].isin(orig_items)].copy()
reused_scales = set(reused['scale']) if len(reused) else set()
scales_to_run = [s for s in orig_items if s not in reused_scales]
print(f"reused GP2_EN1 baseline rows for {len(reused_scales)} scales; "
      f"running the ladder fresh for {len(scales_to_run)}: {scales_to_run}")

with open("log/mylog_2wayCFA_other_gp2_en1_origscales_seed12345.txt", "w") as f:
    with redirect_stdout(f):
        fresh = []
        for scale in scales_to_run:
            items = orig_items[scale]
            # print which scale we are processing through R - this way it doesn't get saved in the log file
            ro.globalenv['scale_to_print'] = scale
            ro.r('print(scale_to_print)')
            # create a neat list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # test (per-scale try/except: one bad scale must not kill the run)
            try:
                cfa_res = run_specific_cfa(
                    whichscale=scale,
                    item_list=items_list,
                    whichcfa='strict',
                    datasets=datasets_runspecific,
                    temp_path=path_to_helpfile,
                    num_iter=num_iter,
                    cpus_to_use=cpus_to_use,
                    return_vals=True
                )
            except Exception as exc:
                record_run_error('baseline', scale, exc)
                continue
            cfa_res = pd.DataFrame(cfa_res)
            cfa_res['scale'] = scale
            fresh.append(cfa_res)
            # persist incrementally so a later crash cannot lose completed scales
            pd.concat([reused] + fresh).to_csv(
                cfa_dir / 'orig_cfa_res_in_progress.csv', index=None)
orig_cfa_res = pd.concat([reused] + fresh, ignore_index=True)
if not len(orig_cfa_res):
    raise RuntimeError('no baseline results (nothing reusable and every '
                       'fresh scale failed); see run_errors.log')

reused GP_EN baseline rows for 0 scales; running the ladder fresh for 6: ['phq_sum', 'gad_sum', 'baars_inattention_sum', 'baars_hyperactivity_sum', 'baars_impulsivity_sum', 'baars_sct_sum']


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




...

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




In [16]:
# convert the p-value columns explicitly (extract_p returns floats for
# tested levels and the string 'NA' for untested ones; the old
# astype(float, errors='ignore') is a silent no-op under pandas 3)
for col in ['pconfig', 'pthresholds', 'pmetric', 'pscalar', 'pstrict']:
    orig_cfa_res[col] = pd.to_numeric(orig_cfa_res[col], errors='coerce')

In [17]:
orig_cfa_res.to_csv(cfa_dir / 'orig_cfa_res.csv', index=None)

## Derive `scales_failing_metric` from the baseline results

Replaces the old hardcoded list — the failure pattern may shift under the corrected measEq models. A scale needs the stepwise metric search unless the `gp2_en1` pair reached metric and passed it (`pmetric` numeric and ≥ .05). Tested levels are floats (`extract_p` reads the exact permutation p from the permuteMeasEq object); levels never reached are recorded as the string `'NA'` (e.g. when configural or thresholds failed), so coerce with `pd.to_numeric(..., errors='coerce')` before filtering; a coerced NaN counts as failing.

In [18]:
# pmetric is numeric (converted above); NaN (level never reached) -> False
metric_pass_by_scale = orig_cfa_res['pmetric'].ge(0.05).groupby(orig_cfa_res['scale']).all()
failing = set(metric_pass_by_scale.index[~metric_pass_by_scale])

# scales VERIFIED metric-invariant on the full item set at baseline -- the
# scalar-continuation loop uses this to decide when the full scale is a
# valid metric core (a scale that ERRORED at baseline is in neither set
# and must not be assumed invariant)
baseline_metric_passers = set(metric_pass_by_scale.index[metric_pass_by_scale])

# keep orig_items order for reproducible loop order
scales_failing_metric = [s for s in orig_items if s in failing]
print(f"{len(scales_failing_metric)} of {len(orig_items)} scales fail gp2_en1 "
      f"metric invariance and go to the stepwise search:")
scales_failing_metric

3 of 6 scales fail gp_en metric invariance and go to the stepwise search:


['phq_sum', 'baars_inattention_sum', 'baars_sct_sum']

## Stepwise metric search (scales failing gp2_en1 metric)

for a set of items:
    if it's gp2_en1 config, gp2_en1 thresholds, and gp2_en1 metric invariant:
        return set of items and list of removed items
    if it's not gp2_en1 config invariant:
        evaluate configural invariance on all k-item ablations (smallest k first; every item is a candidate — no marker under std.lv)
        among ablations that are gp2_en1 config invariant, pick by CFI (0.001 tolerance) → TLI (0.001 tolerance) → RMSEA
        restart the loop with the selected set of items
    if it's gp2_en1 config invariant, but not gp2_en1 thresholds:
        remove the item with the highest aggregated threshold-equality modification index
        restart the loop with the selected set of items
    if it's gp2_en1 thresholds invariant, but not gp2_en1 metric:
        remove the item with the highest aggregated loading modification index
        restart the loop with the selected set of items

In [19]:
# silence R to clean up messages
# be careful doing this, you might miss important warnings
silence_r()

In [20]:
# scales_failing_metric is derived from the baseline results above
stepwise_res = []
histories = []
for scale in scales_failing_metric:
    try:
        final, removed, history = do_three_way_cfa_stepwise_mi(
            scale,
            orig_items=orig_items,
            datasets=datasets_stepwise,
            temp_path=path_to_helpfile,
            num_iter=num_iter,
            cpus_to_use=cpus_to_use,
            min_items=3,
        )
    except Exception as exc:
        # one bad scale must not kill the run; no history pickle is written
        # for this scale, and the scalar loop below treats a metric-failing
        # scale without stepwise output as an error, not a full-scale core
        record_run_error('stepwise_metric', scale, exc)
        continue
    if final is not None:
        final_items = [item_lut[item_no] for item_no in final]
        removed_items = [item_lut[item_no] for item_no in removed]
        row = dict(
            scale=scale,
            item_nos=final,
            removed_nos=removed,
            items=final_items,
            removed=removed_items
        )
    else:
        row = dict(
            scale=scale
        )

    stepwise_res.append(row)
    pd.DataFrame(stepwise_res).to_pickle(cfa_dir / 'stepwise_in_progress.pkl')
    history['scale'] = scale
    history.to_pickle(cfa_dir / f'{scale}_history.pkl')
    histories.append(history)


STEPWISE CFA: PHQ_SUM

--- Iteration 0: 8 items ---
Current: ['phq_1', 'phq_2', 'phq_3', 'phq_4', 'phq_5', 'phq_6', 'phq_7', 'phq_8']
Testing current item set...


Configural failed for at least one comparison; running combinatorial ablation search (all items)...
  -- Ablation level 1 (8 combos) --
  Trying drop of phq_1 -> 7 items


    min_cfi=0.8970 min_tli=0.8455 max_rmsea=0.1835 all_config_passed=False
  Trying drop of phq_2 -> 7 items


    min_cfi=0.9296 min_tli=0.8944 max_rmsea=0.1461 all_config_passed=False
  Trying drop of phq_3 -> 7 items


    min_cfi=0.9189 min_tli=0.8784 max_rmsea=0.1669 all_config_passed=False
  Trying drop of phq_4 -> 7 items


...

    min_cfi=0.9127 min_tli=0.8690 max_rmsea=0.1574 all_config_passed=True
  Trying drop of ('sct_7', 'sct_9') -> 7 items


    min_cfi=0.9494 min_tli=0.9241 max_rmsea=0.1148 all_config_passed=True
  Trying drop of ('sct_8', 'sct_9') -> 7 items


    min_cfi=0.8777 min_tli=0.8165 max_rmsea=0.1918 all_config_passed=False
  Level 2: 4 of 36 combos achieve 3-way configural invariance.
  -> Removing combo ('sct_7', 'sct_9') (ablation_level=2, reason=cfi_max_min)

--- Iteration 1: 7 items ---
Current: ['sct_1', 'sct_2', 'sct_3', 'sct_4', 'sct_5', 'sct_6', 'sct_8']
Testing current item set...


Thresholds OK but metric failed; running loading-MI-based removal...
  Aggregated loading MIs (summed over failing comparisons):
    sct_2: 13.380
    sct_3: 5.540
    sct_8: 2.441
    sct_5: 2.331
    sct_1: 1.074
    sct_6: 0.826
    sct_4: 0.200
  -> Removing: sct_2

--- Iteration 2: 6 items ---
Current: ['sct_1', 'sct_3', 'sct_4', 'sct_5', 'sct_6', 'sct_8']
Testing current item set...



*** 3-way metric invariance achieved with 6 items ***


In [21]:
stepwise_res = pd.DataFrame(stepwise_res)

In [22]:
stepwise_res

,scale,item_nos,removed_nos,items,removed
0,phq_sum,"[phq_1, phq_3, phq_4, phq_6, phq_7]","[phq_2, phq_8, phq_5]","[Little interest or pleasure in doing things, Trouble falling or staying asleep, or sleeping too...","[Feeling down, depressed, irritable or hopeless, Moving or speaking so slowly that other people ..."
1,baars_inattention_sum,"[inattention_3, inattention_4, inattention_5, inattention_6, inattention_7, inattention_8]","[inattention_1, inattention_9, inattention_2]","[Don't listen when spoken to directly, Don't follow through on instructions and fail to finish w...",[Fail to give close attention to details or make careless mistakes in my work or other activitie...
2,baars_sct_sum,"[sct_1, sct_3, sct_4, sct_5, sct_6, sct_8]","[sct_7, sct_9, sct_2]","[Prone to daydreaming when I should be concentrating on something or working, Easily confused, E...","[Underactive or have less energy than others, I don't seem to process information as quickly or ..."


In [23]:
stepwise_res.to_pickle(cfa_dir / 'stepwise.pkl')

## Scalar continuation from the metric cores

The invariance target is **scalar** (thresholds + loadings + intercepts): the GP-vs-enriched latent mean comparisons are only valid under scalar invariance. For every scale we continue from its metric core toward a scalar core; if the continuation bottoms out, the **metric core is that scale's deliverable** (reporting rule: ICCs may use metric-fallback cores, mean comparisons are reported only for scales with scalar cores).

Mechanics under Wu–Estabrook: the scalar delta test is the **param-free omnibus permutation** (W&E fixes group-2 intercepts back to 0 rather than equating them, so `param="intercepts"` has no constraints to point at), and item removal at the scalar level is driven by `lavaan::modindices()` intercept MIs on the scalar fit (score test for freeing each fixed group-2 intercept).

Per scale: `load_metric_run` reads the metric core from this run's pickles (`{scale}_history.pkl`, falling back to `stepwise.pkl` — measEq-era pickles only). Scales with no stepwise metric run (`FileNotFoundError`) passed metric on the full item set at baseline, so the full scale is their metric core. Lower-level permutation tests were already run on exactly these items/data/seeds, so the first iteration assumes them and runs only the scalar test (`assume_metric_invariant=True`, the default); the df ladder is still asserted at every level.

The resulting `stepwise_scalar.pkl` is the **final-cores table** consumed by NB_3_ICC: one row per scale with `core_level` (`'scalar'`, `'metric'`, or `None`), the core item set, and the items removed relative to the original scale.

In [24]:
scalar_stepwise_res = []
scalar_histories = []
for scale in orig_items:
    items_list = orig_items[scale].split("=~", 1)[1].strip().split(" + ")
    try:
        # metric core from this run's measEq-era stepwise pickles
        metric_core, _metric_history = load_metric_run(cfa_dir, scale)
    except FileNotFoundError:
        if scale in baseline_metric_passers:
            # no stepwise metric run exists because the scale passed metric
            # on the full item set at baseline; the full scale is its core
            metric_core = items_list
        else:
            # no stepwise output AND no verified baseline pass: the scale
            # errored upstream -- do NOT assume the full scale is invariant
            record_run_error(
                'scalar_continuation', scale,
                RuntimeError('no metric-stepwise output and no verified '
                             'baseline metric pass; upstream stage errored'))
            history = pd.DataFrame([{
                'iteration': 0, 'phase': 'final', 'n_items': 0,
                'items': tuple(), 'action': 'upstream_error',
            }])
            row = dict(scale=scale, core_level=None, error='upstream_error')
            scalar_stepwise_res.append(row)
            pd.DataFrame(scalar_stepwise_res).to_pickle(
                cfa_dir / 'stepwise_scalar_in_progress.pkl')
            history['scale'] = scale
            history.to_pickle(cfa_dir / f'{scale}_scalar_history.pkl')
            scalar_histories.append(history)
            continue

    if metric_core is None:
        # the stepwise metric search found no invariant core: no deliverable
        print(f"[{scale}] metric search found no invariant core; "
              f"no scalar continuation possible")
        history = pd.DataFrame([{
            'iteration': 0, 'phase': 'final', 'n_items': 0,
            'items': tuple(), 'action': 'no_metric_core',
        }])
        row = dict(scale=scale, core_level=None)
    else:
        try:
            final, removed, history = do_three_way_cfa_stepwise_scalar(
                scale,
                metric_core,
                datasets=datasets_stepwise,
                temp_path=path_to_helpfile,
                num_iter=num_iter,
                cpus_to_use=cpus_to_use,
                min_items=3,
            )
            error = None
        except Exception as exc:
            # the metric core is still a verified deliverable; fall back to
            # it, flag the error, and keep the run alive
            record_run_error('scalar_continuation', scale, exc)
            final, removed = None, []
            error = 'scalar_search_error'
            history = pd.DataFrame([{
                'iteration': 0, 'phase': 'final',
                'n_items': len(metric_core), 'items': tuple(metric_core),
                'action': 'scalar_search_error',
            }])
        if final is not None:
            core, core_level = final, 'scalar'
        else:
            # scalar continuation bottomed out (or errored): the metric
            # core is the scale's deliverable (metric fallback)
            core, core_level = metric_core, 'metric'
        removed_from_orig = [ii for ii in items_list if ii not in core]
        row = dict(
            scale=scale,
            core_level=core_level,
            item_nos=core,
            removed_nos=removed_from_orig,
            items=[item_lut[item_no] for item_no in core],
            removed=[item_lut[item_no] for item_no in removed_from_orig],
            metric_item_nos=metric_core,
            scalar_removed_nos=removed,  # removed during the scalar stage only
            error=error,
        )

    scalar_stepwise_res.append(row)
    pd.DataFrame(scalar_stepwise_res).to_pickle(
        cfa_dir / 'stepwise_scalar_in_progress.pkl')
    history['scale'] = scale
    history.to_pickle(cfa_dir / f'{scale}_scalar_history.pkl')
    scalar_histories.append(history)


STEPWISE SCALAR: PHQ_SUM

--- Iteration 0: 5 items ---
Current: ['phq_1', 'phq_3', 'phq_4', 'phq_6', 'phq_7']
Testing current item set (configural + thresholds + metric assumed from completed metric run; scalar test only)...


Scalar failed; running intercept-MI-based removal...
  Aggregated intercept MIs (summed over failing comparisons):
    phq_7: 11.984
    phq_4: 10.700
    phq_6: 5.277
    phq_3: 2.787
    phq_1: 0.503
  -> Removing: phq_7

--- Iteration 1: 4 items ---
Current: ['phq_1', 'phq_3', 'phq_4', 'phq_6']
Testing current item set up to scalar...


Metric regressed; running loading-MI-based removal...
  Aggregated loading MIs (summed over failing comparisons):
    phq_1: 5.338
    phq_3: 2.276
    phq_4: 1.820
    phq_6: 0.191
  -> Removing: phq_1

--- Iteration 2: 3 items ---
Current: ['phq_3', 'phq_4', 'phq_6']
Testing current item set up to scalar...


Scalar failed; running intercept-MI-based removal...
  Aggregated intercept MIs (summed over failing comparisons):
    phq_6: 18.358
    phq_4: 7.152
    phq_3: 3.204
  -> Removing: phq_6

Hit max_iter (2) without convergence.

STEPWISE SCALAR: GAD_SUM

--- Iteration 0: 7 items ---
Current: ['gad_1', 'gad_2', 'gad_3', 'gad_4', 'gad_5', 'gad_6', 'gad_7']
Testing current item set (configural + thresholds + metric assumed from completed metric run; scalar test only)...



*** 3-way scalar invariance achieved with 7 items ***

STEPWISE SCALAR: BAARS_INATTENTION_SUM

--- Iteration 0: 6 items ---
Current: ['inattention_3', 'inattention_4', 'inattention_5', 'inattention_6', 'inattention_7', 'inattention_8']
Testing current item set (configural + thresholds + metric assumed from completed metric run; scalar test only)...



*** 3-way scalar invariance achieved with 6 items ***

STEPWISE SCALAR: BAARS_HYPERACTIVITY_SUM

--- Iteration 0: 5 items ---
Current: ['hyperactivity_1', 'hyperactivity_2', 'hyperactivity_3', 'hyperactivity_4', 'hyperactivity_5']
Testing current item set (configural + thresholds + metric assumed from completed metric run; scalar test only)...



*** 3-way scalar invariance achieved with 5 items ***

STEPWISE SCALAR: BAARS_IMPULSIVITY_SUM

--- Iteration 0: 4 items ---
Current: ['impulsivity_1', 'impulsivity_2', 'impulsivity_3', 'impulsivity_4']
Testing current item set (configural + thresholds + metric assumed from completed metric run; scalar test only)...



*** 3-way scalar invariance achieved with 4 items ***

STEPWISE SCALAR: BAARS_SCT_SUM

--- Iteration 0: 6 items ---
Current: ['sct_1', 'sct_3', 'sct_4', 'sct_5', 'sct_6', 'sct_8']
Testing current item set (configural + thresholds + metric assumed from completed metric run; scalar test only)...



*** 3-way scalar invariance achieved with 6 items ***


In [25]:
scalar_stepwise_res = pd.DataFrame(scalar_stepwise_res)
scalar_stepwise_res

,scale,core_level,item_nos,removed_nos,items,removed,metric_item_nos,scalar_removed_nos,error
0,phq_sum,metric,"[phq_1, phq_3, phq_4, phq_6, phq_7]","[phq_2, phq_5, phq_8]","[Little interest or pleasure in doing things, Trouble falling or staying asleep, or sleeping too...","[Feeling down, depressed, irritable or hopeless, Poor appetite or overeating, Moving or speaking...","[phq_1, phq_3, phq_4, phq_6, phq_7]","[phq_7, phq_1, phq_6]",None
1,gad_sum,scalar,"[gad_1, gad_2, gad_3, gad_4, gad_5, gad_6, gad_7]",[],"[Feeling nervous, anxious, or on edge, Not being able to stop or control worrying, Worrying too ...",[],"[gad_1, gad_2, gad_3, gad_4, gad_5, gad_6, gad_7]",[],None
2,baars_inattention_sum,scalar,"[inattention_3, inattention_4, inattention_5, inattention_6, inattention_7, inattention_8]","[inattention_1, inattention_2, inattention_9]","[Don't listen when spoken to directly, Don't follow through on instructions and fail to finish w...",[Fail to give close attention to details or make careless mistakes in my work or other activitie...,"[inattention_3, inattention_4, inattention_5, inattention_6, inattention_7, inattention_8]",[],None
3,baars_hyperactivity_sum,scalar,"[hyperactivity_1, hyperactivity_2, hyperactivity_3, hyperactivity_4, hyperactivity_5]",[],"[Fidget with hands or feet or squirm in seat, Leave my seat in classrooms or in other situations...",[],"[hyperactivity_1, hyperactivity_2, hyperactivity_3, hyperactivity_4, hyperactivity_5]",[],None
4,baars_impulsivity_sum,scalar,"[impulsivity_1, impulsivity_2, impulsivity_3, impulsivity_4]",[],"[Talk excessively (in social situations), Blurt out answers before questions have been completed...",[],"[impulsivity_1, impulsivity_2, impulsivity_3, impulsivity_4]",[],None
5,baars_sct_sum,scalar,"[sct_1, sct_3, sct_4, sct_5, sct_6, sct_8]","[sct_2, sct_7, sct_9]","[Prone to daydreaming when I should be concentrating on something or working, Easily confused, E...","[Have trouble staying alert or awake in boring situations, Underactive or have less energy than ...","[sct_1, sct_3, sct_4, sct_5, sct_6, sct_8]",[],None


In [26]:
scalar_stepwise_res.to_pickle(cfa_dir / 'stepwise_scalar.pkl')

## Strict invariance report on the final cores

Report-only (per the handoff's reporting rule): each scale's final core (scalar core, or metric fallback) is run up the full ladder to **strict** on the `gp2_en1` pair. Strict pass/fail is reported in the manuscript but drives no item removal. The strict delta test is the param-free omnibus permutation (same W&E fixing logic as scalar).

In [27]:
with open(log_dir / "mylog_2wayCFA_other_gp2_en1_finalcores_strict_seed12345.txt", "w") as f:
    with redirect_stdout(f):
        strict_report = []
        for row in scalar_stepwise_res.itertuples():
            if row.core_level not in ('scalar', 'metric'):
                continue
            try:
                res = run_specific_cfa(
                    whichscale=row.scale,
                    item_list=list(row.item_nos),
                    whichcfa='strict',
                    datasets=datasets_runspecific,
                    temp_path=path_to_helpfile,
                    num_iter=num_iter,
                    cpus_to_use=cpus_to_use,
                    return_vals=True,
                )
            except Exception as exc:
                record_run_error('strict_report', row.scale, exc)
                continue
            res = pd.DataFrame(res)
            res['scale'] = row.scale
            res['core_level'] = row.core_level
            strict_report.append(res)
            # persist incrementally
            pd.concat(strict_report).replace("NA", pd.NA).to_csv(
                cfa_dir / 'final_cores_strict_report_in_progress.csv',
                index=None)
strict_report = pd.concat(strict_report) if strict_report else pd.DataFrame()
strict_report = strict_report.replace("NA", pd.NA)
strict_report.to_csv(cfa_dir / 'final_cores_strict_report.csv', index=None)
strict_report

,pair,pconfig,pthresholds,pmetric,pscalar,pstrict,scale,core_level
0,GP_EN,0.388,0.374,0.067,0.000,<NA>,phq_sum,metric
0,GP_EN,0.065,0.280,0.301,0.244,0.056,gad_sum,scalar
0,GP_EN,0.277,0.111,0.790,0.107,0.229,baars_inattention_sum,scalar
0,GP_EN,0.147,0.676,0.415,0.229,0.084,baars_hyperactivity_sum,scalar
0,GP_EN,0.481,0.376,0.352,0.310,0.704,baars_impulsivity_sum,scalar
0,GP_EN,0.050,0.186,0.123,0.069,0.924,baars_sct_sum,scalar


In [28]:
# ---- Overnight run summary ----
print(f"scales in baseline results:      "
      f"{orig_cfa_res['scale'].nunique()} / {len(orig_items)}")
print(f"scales failing gp2_en1 metric:     {len(scales_failing_metric)}")
n_scalar = int((scalar_stepwise_res.core_level == 'scalar').sum())
n_metric = int((scalar_stepwise_res.core_level == 'metric').sum())
n_none = int(scalar_stepwise_res.core_level.isnull().sum())
print(f"final cores: {n_scalar} scalar, {n_metric} metric fallback, "
      f"{n_none} without an invariant core")
if run_error_records:
    print(f"\n!!! {len(run_error_records)} ERROR(S) recorded during this run "
          f"(details + tracebacks in {cfa_dir / 'run_errors.log'}):")
    for rec in run_error_records:
        print(f"  [{rec['stage']}] {rec['scale']}: {rec['error']}")
else:
    print("\nno errors recorded during this run")

scales in baseline results:      6 / 6
scales failing gp_en metric:     3
final cores: 5 scalar, 1 metric fallback, 0 without an invariant core

no errors recorded during this run
